# 🎙️ Haru-4B: Live Voice Portal & Quantized Model Testing

Test **[Jayanth-MKV/Haru-4B-GGUF](https://huggingface.co/Jayanth-MKV/Haru-4B-GGUF)** and **[higurashi-ai/Haru-4B](https://huggingface.co/higurashi-ai/Haru-4B)** on **Google Colab** (Free T4 GPU).

### ⚡ Ensure GPU is Selected:
Click **Runtime** > **Change runtime type** > Select **T4 GPU** > **Save**.

In [ ]:
# Step 1: Verify GPU
!nvidia-smi

In [ ]:
# Step 2: Clone Haru & Install Dependencies (Notice: must cd into Haru/haru)
!rm -rf /content/Haru
!git clone https://github.com/higurashi-ai/Haru.git /content/Haru
%cd /content/Haru/haru
!pip install -q -e ".[share]"
!pip install -q huggingface_hub bitsandbytes accelerate

---
## 🧪 Part 1: Test Our Quantized Model Backbone (Jayanth-MKV/Haru-4B-GGUF)

Download and run our 4-bit GGUF model (`qwen-haru-q4_k_m.gguf`, 2.6 GB) on the T4 GPU via `llama.cpp`:

In [ ]:
# Build llama.cpp with CUDA support
!git clone https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
%cd /content/llama.cpp
!cmake -B build -DGGML_CUDA=ON > /dev/null 2>&1
!cmake --build build --config Release -j 4 > /dev/null 2>&1

# Download our Quantized GGUF model from Hugging Face (2.6 GB)
from huggingface_hub import hf_hub_download
print("Downloading qwen-haru-q4_k_m.gguf from Jayanth-MKV/Haru-4B-GGUF...")
gguf_path = hf_hub_download(repo_id="Jayanth-MKV/Haru-4B-GGUF", filename="qwen-haru-q4_k_m.gguf")
print("Downloaded successfully to:", gguf_path)

# Run high-speed GPU inference (-ngl 99 offloads all 32 layers to VRAM)
!./build/bin/llama-cli -m "{gguf_path}" -p "Hello! How are you doing today? Let's have a conversation." -n 128 -ngl 99

---
## 🎙️ Part 2: Launch the Full-Duplex Live Voice Portal (Port 8998 with --share)

Launches the continuous full-duplex spoken dialogue server with a public HTTPS link (`https://xxxx.gradio.live`):

In [ ]:
%cd /content/Haru/haru
# --warmup-frames 0 loads the model in 3 seconds
# --share gives you a public HTTPS URL (https://xxxx.gradio.live) to use your microphone
!python -m haru.server --share --warmup-frames 0 --host 0.0.0.0 --port 8998